# Exercise Rep Counter Model using YOLOv8 Nano

This notebook implements a rep counter that:
- Uses YOLOv8 Nano Pose for pose detection
- Trains a classifier to recognize start/end positions for different exercises
- Counts reps by tracking state transitions (start → end → start)
- Works with multiple exercise classes

## Directory Structure Expected:
```
reptraining/
├── exercise_name_1/
│   ├── start/
│   │   ├── img1.jpg
│   │   └── ...
│   └── end/
│       ├── img1.jpg
│       └── ...
├── exercise_name_2/
│   └── ...
```

## 1. Install Dependencies

In [ ]:
!pip install ultralytics opencv-python numpy scikit-learn joblib torch

## 2. Import Libraries

In [ ]:
import os
import cv2
import numpy as np
from ultralytics import YOLO
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import joblib
from pathlib import Path
import json
from collections import deque
import warnings
warnings.filterwarnings('ignore')

## 3. Configuration

In [ ]:
# Paths
REPTRAINING_DIR = Path("reptraining")
MODEL_OUTPUT_DIR = Path("output/rep_counter_model")
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Model configuration
POSE_MODEL_PATH = "../yolov8n-pose.pt"  # YOLOv8 Nano Pose model
CLASSIFIER_PATH = MODEL_OUTPUT_DIR / "rep_phase_classifier.pkl"
LABEL_ENCODER_PATH = MODEL_OUTPUT_DIR / "label_encoder.pkl"
METADATA_PATH = MODEL_OUTPUT_DIR / "model_metadata.json"

# Training parameters
TEST_SIZE = 0.2
RANDOM_STATE = 42

print(f"Training data directory: {REPTRAINING_DIR}")
print(f"Model output directory: {MODEL_OUTPUT_DIR}")

## 4. Load YOLOv8 Nano Pose Model

In [ ]:
# Load YOLOv8 Nano Pose model
print("Loading YOLOv8 Nano Pose model...")
pose_model = YOLO(POSE_MODEL_PATH)
print("YOLOv8 Nano Pose model loaded successfully!")

## 5. Helper Functions for Pose Feature Extraction

In [ ]:
def extract_pose_keypoints(image_path, pose_model):
    """
    Extract pose keypoints from an image using YOLOv8 Pose.
    
    Returns:
        - Flattened keypoints array (x, y, confidence for each keypoint)
        - None if no person detected
    """
    img = cv2.imread(str(image_path))
    if img is None:
        return None
    
    # Run pose detection
    results = pose_model(img, verbose=False)
    
    # Check if any person detected
    if len(results) == 0 or results[0].keypoints is None:
        return None
    
    # Get keypoints for the first detected person
    keypoints = results[0].keypoints.data
    
    if len(keypoints) == 0:
        return None
    
    # Take the first person's keypoints and flatten
    # YOLOv8 Pose returns (N, 17, 3) where N is number of people, 17 is keypoints, 3 is (x, y, confidence)
    kp = keypoints[0].cpu().numpy().flatten()
    
    return kp


def normalize_keypoints(keypoints):
    """
    Normalize keypoints to be scale and translation invariant.
    """
    if keypoints is None or len(keypoints) == 0:
        return None
    
    # Reshape to (17, 3) format
    kp = keypoints.reshape(-1, 3)
    
    # Extract x, y coordinates (ignore confidence for normalization)
    xy = kp[:, :2]
    conf = kp[:, 2]
    
    # Calculate bounding box
    valid_points = xy[conf > 0.5]
    
    if len(valid_points) == 0:
        return keypoints
    
    # Normalize by bounding box
    min_xy = valid_points.min(axis=0)
    max_xy = valid_points.max(axis=0)
    range_xy = max_xy - min_xy
    
    # Avoid division by zero
    range_xy[range_xy == 0] = 1
    
    # Normalize x, y coordinates
    xy_normalized = (xy - min_xy) / range_xy
    
    # Reconstruct keypoints with normalized coordinates
    kp_normalized = np.column_stack([xy_normalized, conf])
    
    return kp_normalized.flatten()

## 6. Load and Prepare Training Data

In [ ]:
def load_training_data(data_dir, pose_model):
    """
    Load training data from directory structure:
    data_dir/
        exercise_name/
            start/
                img1.jpg, img2.jpg, ...
            end/
                img1.jpg, img2.jpg, ...
    
    Returns:
        X: Feature matrix (keypoints)
        y: Labels (exercise_name_phase, e.g., "pushup_start", "pushup_end")
        exercise_classes: List of exercise names
    """
    X = []
    y = []
    exercise_classes = []
    
    if not data_dir.exists():
        print(f"Warning: {data_dir} does not exist!")
        return np.array([]), np.array([]), []
    
    # Iterate through exercise folders
    exercise_folders = [f for f in data_dir.iterdir() if f.is_dir()]
    
    if len(exercise_folders) == 0:
        print(f"Warning: No exercise folders found in {data_dir}!")
        return np.array([]), np.array([]), []
    
    print(f"Found {len(exercise_folders)} exercise classes")
    
    for exercise_folder in exercise_folders:
        exercise_name = exercise_folder.name
        exercise_classes.append(exercise_name)
        print(f"\nProcessing exercise: {exercise_name}")
        
        # Process start and end phases
        for phase in ['start', 'end']:
            phase_dir = exercise_folder / phase
            
            if not phase_dir.exists():
                print(f"  Warning: {phase} folder not found for {exercise_name}")
                continue
            
            # Get all image files
            image_files = list(phase_dir.glob('*.jpg')) + list(phase_dir.glob('*.png')) + list(phase_dir.glob('*.jpeg'))
            
            print(f"  Processing {len(image_files)} {phase} images...")
            
            for img_path in image_files:
                # Extract keypoints
                keypoints = extract_pose_keypoints(img_path, pose_model)
                
                if keypoints is not None:
                    # Normalize keypoints
                    keypoints_norm = normalize_keypoints(keypoints)
                    
                    if keypoints_norm is not None:
                        X.append(keypoints_norm)
                        y.append(f"{exercise_name}_{phase}")
    
    print(f"\nTotal samples collected: {len(X)}")
    
    return np.array(X), np.array(y), exercise_classes


# Load the training data
print("Loading training data...")
X, y, exercise_classes = load_training_data(REPTRAINING_DIR, pose_model)

if len(X) > 0:
    print(f"\nDataset shape: {X.shape}")
    print(f"Number of labels: {len(y)}")
    print(f"Exercise classes: {exercise_classes}")
    print(f"Label distribution:")
    unique, counts = np.unique(y, return_counts=True)
    for label, count in zip(unique, counts):
        print(f"  {label}: {count}")
else:
    print("\nNo training data loaded. Please add images to the reptraining folder.")

## 7. Train Phase Classifier

This classifier will predict whether a pose is at the "start" or "end" phase of an exercise.

In [ ]:
if len(X) > 0:
    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )
    
    print(f"Training set size: {len(X_train)}")
    print(f"Test set size: {len(X_test)}")
    
    # Train Random Forest Classifier
    print("\nTraining Random Forest Classifier...")
    classifier = RandomForestClassifier(
        n_estimators=200,
        max_depth=20,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=1
    )
    
    classifier.fit(X_train, y_train)
    print("Training complete!")
    
    # Evaluate the model
    print("\n" + "="*50)
    print("Model Evaluation")
    print("="*50)
    
    y_pred = classifier.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    
    print(f"\nAccuracy: {accuracy:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    
    # Save the model
    print(f"\nSaving model to {CLASSIFIER_PATH}...")
    joblib.dump(classifier, CLASSIFIER_PATH)
    
    # Save metadata
    metadata = {
        "exercise_classes": exercise_classes,
        "phase_labels": list(np.unique(y)),
        "num_features": X.shape[1],
        "accuracy": float(accuracy),
        "num_train_samples": len(X_train),
        "num_test_samples": len(X_test)
    }
    
    with open(METADATA_PATH, 'w') as f:
        json.dump(metadata, f, indent=4)
    
    print(f"Metadata saved to {METADATA_PATH}")
    print("\nModel training complete!")
    
else:
    print("Cannot train model without data. Please add training images first.")

## 8. Rep Counter Class

This class implements the rep counting logic by tracking state transitions from start → end → start.

In [ ]:
class RepCounter:
    """
    Rep counter that tracks exercise phases and counts reps.
    """
    
    def __init__(self, pose_model, classifier, smoothing_window=5, confidence_threshold=0.6):
        """
        Args:
            pose_model: YOLOv8 Pose model
            classifier: Trained phase classifier
            smoothing_window: Number of frames to use for smoothing predictions
            confidence_threshold: Minimum confidence to accept predictions
        """
        self.pose_model = pose_model
        self.classifier = classifier
        self.smoothing_window = smoothing_window
        self.confidence_threshold = confidence_threshold
        
        # State tracking
        self.current_exercise = None
        self.current_phase = None
        self.rep_count = 0
        self.phase_history = deque(maxlen=smoothing_window)
        
        # Statistics
        self.frame_count = 0
        self.last_phase_change_frame = 0
        
    def reset(self, exercise_name=None):
        """Reset the rep counter."""
        self.current_exercise = exercise_name
        self.current_phase = None
        self.rep_count = 0
        self.phase_history.clear()
        self.frame_count = 0
        self.last_phase_change_frame = 0
    
    def predict_phase(self, frame):
        """
        Predict the exercise phase from a video frame.
        
        Returns:
            phase_label: Predicted phase (e.g., "pushup_start" or "pushup_end")
            confidence: Prediction confidence
        """
        # Extract keypoints
        keypoints = extract_pose_keypoints_from_frame(frame, self.pose_model)
        
        if keypoints is None:
            return None, 0.0
        
        # Normalize keypoints
        keypoints_norm = normalize_keypoints(keypoints)
        
        if keypoints_norm is None:
            return None, 0.0
        
        # Predict phase
        phase_label = self.classifier.predict([keypoints_norm])[0]
        
        # Get prediction probabilities for confidence
        proba = self.classifier.predict_proba([keypoints_norm])[0]
        confidence = proba.max()
        
        return phase_label, confidence
    
    def update(self, frame):
        """
        Update rep counter with a new frame.
        
        Returns:
            dict with current state information
        """
        self.frame_count += 1
        
        # Predict phase
        phase_label, confidence = self.predict_phase(frame)
        
        if phase_label is None or confidence < self.confidence_threshold:
            return {
                'rep_count': self.rep_count,
                'current_phase': self.current_phase,
                'confidence': 0.0,
                'status': 'no_detection'
            }
        
        # Add to history for smoothing
        self.phase_history.append(phase_label)
        
        # Use majority vote for smoothing
        if len(self.phase_history) >= self.smoothing_window:
            unique, counts = np.unique(list(self.phase_history), return_counts=True)
            smoothed_phase = unique[counts.argmax()]
        else:
            smoothed_phase = phase_label
        
        # Extract exercise name and phase from label
        if '_' in smoothed_phase:
            exercise, phase = smoothed_phase.rsplit('_', 1)
        else:
            exercise, phase = smoothed_phase, 'unknown'
        
        # Set current exercise if not set
        if self.current_exercise is None:
            self.current_exercise = exercise
        
        # Check for phase transition
        status = 'in_progress'
        
        if self.current_phase != phase:
            # Prevent rapid transitions (debouncing)
            if self.frame_count - self.last_phase_change_frame > 5:
                # Count a rep when transitioning from end back to start
                if self.current_phase == 'end' and phase == 'start':
                    self.rep_count += 1
                    status = 'rep_counted'
                
                self.current_phase = phase
                self.last_phase_change_frame = self.frame_count
                status = 'phase_changed'
        
        return {
            'rep_count': self.rep_count,
            'current_phase': self.current_phase,
            'current_exercise': self.current_exercise,
            'confidence': confidence,
            'status': status,
            'raw_phase': phase_label
        }


def extract_pose_keypoints_from_frame(frame, pose_model):
    """
    Extract pose keypoints from a video frame.
    """
    # Run pose detection
    results = pose_model(frame, verbose=False)
    
    # Check if any person detected
    if len(results) == 0 or results[0].keypoints is None:
        return None
    
    # Get keypoints for the first detected person
    keypoints = results[0].keypoints.data
    
    if len(keypoints) == 0:
        return None
    
    # Take the first person's keypoints and flatten
    kp = keypoints[0].cpu().numpy().flatten()
    
    return kp


print("RepCounter class defined!")

## 9. Load Trained Model (for Inference)

In [ ]:
# Load the trained classifier
if CLASSIFIER_PATH.exists():
    print("Loading trained classifier...")
    trained_classifier = joblib.load(CLASSIFIER_PATH)
    print("Classifier loaded successfully!")
    
    # Load metadata
    if METADATA_PATH.exists():
        with open(METADATA_PATH, 'r') as f:
            metadata = json.load(f)
        print("\nModel Metadata:")
        print(f"  Exercise classes: {metadata['exercise_classes']}")
        print(f"  Accuracy: {metadata['accuracy']:.4f}")
        print(f"  Training samples: {metadata['num_train_samples']}")
else:
    print("No trained model found. Please train the model first.")
    trained_classifier = None

## 10. Test on Video

Test the rep counter on a video file.

In [ ]:
def test_rep_counter_on_video(video_path, output_path=None, display=False):
    """
    Test the rep counter on a video file.
    
    Args:
        video_path: Path to input video
        output_path: Path to save output video with annotations (optional)
        display: Whether to display the video in real-time
    """
    if trained_classifier is None:
        print("Error: No trained classifier available!")
        return
    
    # Initialize rep counter
    rep_counter = RepCounter(pose_model, trained_classifier)
    
    # Open video
    cap = cv2.VideoCapture(str(video_path))
    
    if not cap.isOpened():
        print(f"Error: Could not open video {video_path}")
        return
    
    # Get video properties
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f"\nProcessing video: {video_path}")
    print(f"Resolution: {width}x{height}")
    print(f"FPS: {fps}")
    print(f"Total frames: {total_frames}")
    
    # Setup video writer if output path provided
    writer = None
    if output_path:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        writer = cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))
    
    frame_idx = 0
    
    try:
        while True:
            ret, frame = cap.read()
            
            if not ret:
                break
            
            frame_idx += 1
            
            # Update rep counter
            result = rep_counter.update(frame)
            
            # Annotate frame
            annotated_frame = frame.copy()
            
            # Draw rep count
            cv2.putText(
                annotated_frame,
                f"Reps: {result['rep_count']}",
                (30, 60),
                cv2.FONT_HERSHEY_SIMPLEX,
                2,
                (0, 255, 0),
                3
            )
            
            # Draw current phase
            if result['current_phase']:
                phase_text = f"Phase: {result['current_phase'].upper()}"
                color = (0, 255, 255) if result['current_phase'] == 'start' else (255, 0, 255)
                cv2.putText(
                    annotated_frame,
                    phase_text,
                    (30, 120),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.2,
                    color,
                    2
                )
            
            # Draw exercise name
            if result['current_exercise']:
                cv2.putText(
                    annotated_frame,
                    f"Exercise: {result['current_exercise']}",
                    (30, 170),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (255, 255, 255),
                    2
                )
            
            # Draw confidence
            cv2.putText(
                annotated_frame,
                f"Confidence: {result['confidence']:.2f}",
                (30, 220),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (200, 200, 200),
                2
            )
            
            # Draw pose skeleton
            pose_results = pose_model(frame, verbose=False)
            if len(pose_results) > 0 and pose_results[0].keypoints is not None:
                keypoints = pose_results[0].keypoints.data
                if len(keypoints) > 0:
                    kp = keypoints[0].cpu().numpy()
                    
                    # Draw keypoints
                    for i in range(len(kp)):
                        x, y, conf = kp[i]
                        if conf > 0.5:
                            cv2.circle(annotated_frame, (int(x), int(y)), 5, (0, 255, 0), -1)
            
            # Write frame
            if writer:
                writer.write(annotated_frame)
            
            # Display frame (optional)
            if display:
                cv2.imshow('Rep Counter', annotated_frame)
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break
            
            # Progress indicator
            if frame_idx % 30 == 0:
                progress = (frame_idx / total_frames) * 100
                print(f"Progress: {progress:.1f}% - Reps: {result['rep_count']}", end='\r')
    
    finally:
        cap.release()
        if writer:
            writer.release()
        if display:
            cv2.destroyAllWindows()
    
    print(f"\n\nProcessing complete!")
    print(f"Total reps counted: {rep_counter.rep_count}")
    
    if output_path:
        print(f"Output saved to: {output_path}")
    
    return rep_counter.rep_count


print("Video testing function ready!")

## 11. Example: Test on a Video File

Replace the video path below with your test video.

In [ ]:
# Example usage - modify the video path to test on your video
VIDEO_PATH = "../data/videos/pushups/example_video.mp4"  # Change this to your video path
OUTPUT_PATH = "output/rep_counter_test_output.mp4"

# Uncomment the line below to test on a video
# rep_count = test_rep_counter_on_video(VIDEO_PATH, output_path=OUTPUT_PATH, display=False)

print("To test on a video, uncomment the line above and set the correct video path.")

## 12. Utility: Create Training Folder Structure

Run this cell to create the folder structure for training data.

In [ ]:
def create_training_structure(exercise_names):
    """
    Create the folder structure for training data.
    
    Args:
        exercise_names: List of exercise names
    """
    for exercise in exercise_names:
        start_dir = REPTRAINING_DIR / exercise / "start"
        end_dir = REPTRAINING_DIR / exercise / "end"
        
        start_dir.mkdir(parents=True, exist_ok=True)
        end_dir.mkdir(parents=True, exist_ok=True)
        
        print(f"Created: {start_dir}")
        print(f"Created: {end_dir}")
    
    print("\nFolder structure created successfully!")
    print("\nNext steps:")
    print("1. Add images of the START pose to the 'start' folders")
    print("2. Add images of the END pose to the 'end' folders")
    print("3. Re-run the training cells (sections 6-7)")


# Example: Create structure for common exercises
# Uncomment and modify the list below to create your exercise folders
# exercise_list = ["pushups", "squats", "lunges", "jumping_jacks"]
# create_training_structure(exercise_list)

print("To create training folders, uncomment and modify the lines above.")

---

## 📋 Usage Summary

### Training the Model:

1. **Prepare training data:**
   - Create folder structure using section 12
   - Add images showing START poses to `reptraining/[exercise]/start/`
   - Add images showing END poses to `reptraining/[exercise]/end/`
   - Images should contain clear views of the person performing the exercise

2. **Train the model:**
   - Run sections 1-7 to train the phase classifier
   - The model will be saved to `output/rep_counter_model/`

3. **Test the model:**
   - Run section 9 to load the trained model
   - Run section 10-11 to test on video files

### How it Works:

- **YOLOv8 Nano Pose**: Detects 17 keypoints on the person's body
- **Phase Classifier**: A Random Forest classifier that learns start/end poses for each exercise
- **Rep Counter**: Tracks state transitions (start → end → start) to count reps
- **Smoothing**: Uses a sliding window to smooth predictions and reduce noise

### Key Features:

✅ Works with multiple exercise classes  
✅ Uses YOLOv8 Nano (not Mediapipe)  
✅ Real-time rep counting capability  
✅ Video testing with visual annotations  
✅ Confidence scoring and smoothing  
✅ Debouncing to prevent false counts  

### Tips for Better Performance:

- Use high-quality images with clear poses
- Have 20-50+ images per phase for each exercise
- Ensure consistent lighting and camera angles
- Test on videos similar to your training data
- Adjust `smoothing_window` and `confidence_threshold` in RepCounter if needed

## 13. Advanced: Custom RepCounter Configuration

You can customize the RepCounter behavior by adjusting these parameters:

In [ ]:
# Example: Create a custom rep counter with different parameters
if trained_classifier is not None:
    custom_rep_counter = RepCounter(
        pose_model=pose_model,
        classifier=trained_classifier,
        smoothing_window=7,          # Increase for more smoothing (less noise, but slower response)
        confidence_threshold=0.7     # Increase to be more strict about predictions
    )
    
    print("Custom RepCounter created with:")
    print(f"  - Smoothing window: {custom_rep_counter.smoothing_window} frames")
    print(f"  - Confidence threshold: {custom_rep_counter.confidence_threshold}")
    print("\nYou can use this custom_rep_counter in place of the default one.")
else:
    print("Train the model first before creating a custom RepCounter.")

## 14. Optional: Real-time Webcam Testing

Test the rep counter with your webcam in real-time.

In [ ]:
def test_rep_counter_webcam(camera_index=0):
    """
    Test the rep counter using webcam in real-time.
    Press 'q' to quit, 'r' to reset counter.
    
    Args:
        camera_index: Camera device index (usually 0 for default webcam)
    """
    if trained_classifier is None:
        print("Error: No trained classifier available!")
        return
    
    # Initialize rep counter
    rep_counter = RepCounter(pose_model, trained_classifier)
    
    # Open webcam
    cap = cv2.VideoCapture(camera_index)
    
    if not cap.isOpened():
        print(f"Error: Could not open camera {camera_index}")
        return
    
    print("\n" + "="*50)
    print("Real-time Rep Counter")
    print("="*50)
    print("Press 'q' to quit")
    print("Press 'r' to reset counter")
    print("="*50 + "\n")
    
    try:
        while True:
            ret, frame = cap.read()
            
            if not ret:
                print("Error: Could not read frame from camera")
                break
            
            # Update rep counter
            result = rep_counter.update(frame)
            
            # Create display frame
            display_frame = frame.copy()
            
            # Draw rep count (large and prominent)
            cv2.putText(
                display_frame,
                f"REPS: {result['rep_count']}",
                (30, 80),
                cv2.FONT_HERSHEY_SIMPLEX,
                2.5,
                (0, 255, 0),
                4
            )
            
            # Draw current phase
            if result['current_phase']:
                phase_text = result['current_phase'].upper()
                color = (0, 255, 255) if result['current_phase'] == 'start' else (255, 0, 255)
                cv2.putText(
                    display_frame,
                    f"Phase: {phase_text}",
                    (30, 150),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.5,
                    color,
                    3
                )
            
            # Draw exercise name
            if result['current_exercise']:
                cv2.putText(
                    display_frame,
                    f"Exercise: {result['current_exercise']}",
                    (30, 200),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (255, 255, 255),
                    2
                )
            
            # Draw confidence
            conf_color = (0, 255, 0) if result['confidence'] > 0.7 else (0, 165, 255)
            cv2.putText(
                display_frame,
                f"Confidence: {result['confidence']:.2f}",
                (30, 250),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                conf_color,
                2
            )
            
            # Draw status indicator
            if result['status'] == 'rep_counted':
                cv2.putText(
                    display_frame,
                    "REP COUNTED!",
                    (display_frame.shape[1] // 2 - 150, display_frame.shape[0] // 2),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.5,
                    (0, 255, 0),
                    3
                )
            
            # Draw pose skeleton
            pose_results = pose_model(frame, verbose=False)
            if len(pose_results) > 0 and pose_results[0].keypoints is not None:
                keypoints = pose_results[0].keypoints.data
                if len(keypoints) > 0:
                    kp = keypoints[0].cpu().numpy()
                    
                    # Draw keypoints
                    for i in range(len(kp)):
                        x, y, conf = kp[i]
                        if conf > 0.5:
                            cv2.circle(display_frame, (int(x), int(y)), 6, (0, 255, 0), -1)
                            cv2.circle(display_frame, (int(x), int(y)), 8, (255, 255, 255), 2)
            
            # Draw instructions
            cv2.putText(
                display_frame,
                "Press 'q' to quit | 'r' to reset",
                (30, display_frame.shape[0] - 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (200, 200, 200),
                2
            )
            
            # Display frame
            cv2.imshow('Rep Counter - Webcam', display_frame)
            
            # Handle key presses
            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):
                break
            elif key == ord('r'):
                rep_counter.reset()
                print("Counter reset!")
    
    finally:
        cap.release()
        cv2.destroyAllWindows()
    
    print(f"\nFinal rep count: {rep_counter.rep_count}")
    print("Webcam testing ended.")


# Uncomment the line below to start webcam testing
# test_rep_counter_webcam()

print("To start webcam testing, uncomment the line above.")